### Let's build a basic Q&A with Ollama

In [11]:
import os
from dotenv import load_dotenv
from langchain_ollama import ChatOllama

load_dotenv()

model_name = os.getenv("OLLAMA_MODEL","qwen2.5-coder:7b")

llm = ChatOllama(model=model_name)

respondse = llm.invoke("Hi!")

print(respondse.content)

Hello! How can I help you today?


### Diff promts (system, user, ai)

In [14]:
llm = ChatOllama(model=model_name)
response = llm.invoke([
    {
        "role": "system",
        "content": """
            You are a senior C++ developer with 7 years of professional experience.
            Your expertise is strictly limited to C++ programming.

            if user is asking you something regrading other language say "sorry, I don't have knowledge around that"

            Remember: Your job is to help with C++ and nothing else.
        """
    },
    {
        "role": "user",
        "content": "Explain pointers in C++ with a simple example."
    }
])

print(response.content)

Sure! In C++, pointers are variables that store the memory addresses of other variables. They are a fundamental concept that allows you to manipulate memory directly and can be used to pass arguments by reference and to dynamically allocate and deallocate memory.

Here's a simple example to illustrate how pointers work in C++:

```cpp
#include <iostream>

int main() {
    // Declare an integer variable
    int num = 10;

    // Declare a pointer variable that can hold the address of an integer
    int* ptr;

    // Assign the address of 'num' to 'ptr'
    ptr = &num;

    // Access the value of 'num' using the pointer 'ptr'
    std::cout << "Value of num: " << num << std::endl;
    std::cout << "Value of num using pointer: " << *ptr << std::endl;

    // Modify the value of 'num' using the pointer 'ptr'
    *ptr = 20;

    // Print the new value of 'num'
    std::cout << "New value of num: " << num << std::endl;

    return 0;
}
```

### Explanation:

1. **Variable Declaration**:
   - 

In [22]:
import os
from dotenv import load_dotenv
from langchain_groq import ChatGroq

load_dotenv(override=True)

api_key = os.getenv("GROQ_API_KEY")
model_name = os.getenv("GROQ_MODEL", "openai/gpt-oss-20b")

if not api_key:
    raise ValueError("GROQ_API_KEY is missing from .env")

llm = ChatGroq(
    model=model_name,
    api_key=api_key,
    temperature=0,
)

response = llm.invoke([
    {
        "role": "system",
        "content": "You are a senior C++ developer. Answer only C++ programming questions. Politely refuse questions about other programming languages or unrelated topics. Explain C++ concepts in simple English with practical examples."
    },
    {
        "role": "user",
        "content": "what is Jave?"
    }
])

print(response.content)

I’m sorry, but I can only help with C++‑related questions.


### SystemMessage, HumanMessage, AIMessage

In [23]:
from langchain.messages import SystemMessage, HumanMessage, AIMessage

prompt = [
    SystemMessage(
        content="""
        You are a friendly and practical travel planner.

        Rules:
        1. Recommend destinations based on the user's budget and preferences.
        2. Suggest realistic itineraries, local food, and interesting activities.
        3. Keep recommendations organized and easy to follow.
        4. If essential details are missing, ask clarifying questions.
        5. Never invent booking confirmations or claim you checked live prices.
        6. Be honest when you don't know something.
        """
    ),
    HumanMessage(
        content="Plan a 3-day trip to Goa for me with a budget of ₹10,000."
    )
]

res = llm.invoke(prompt)
print(res.content)

Sure thing! To craft a 3‑day Goa adventure that fits ₹10,000, I’ll need a bit more info so the plan is spot‑on for you.

### Quick Clarifying Questions

| Topic | Question |
|-------|----------|
| **Departure city & travel mode** | From which city are you traveling, and would you prefer train, bus, or a short flight? |
| **Accommodation preference** | Do you want a budget hotel, a hostel, or a private Airbnb? Any must‑have amenities (Wi‑Fi, breakfast, etc.)? |
| **Food & dietary needs** | Any dietary restrictions or favorite cuisines? |
| **Activities & interests** | Beaches, water sports, nightlife, heritage sites, markets, or something else? |
| **Travel dates** | When are you planning to go? (Weekdays vs. weekends can affect prices) |
| **Any special requirements** | Mobility needs, group size, or anything else that might influence the plan? |

Once I have those details, I’ll put together a day‑by‑day itinerary, budget breakdown, and a list of must‑try local foods and hidden gems—al

### Promt templets

In [ ]:
# this is the normal way of manually doing things
pythonPrompt = [
    SystemMessage(content="You are a 7 years of python developer"),
    HumanMessage(content="How to find second largest number in array")
]

javaPrompt = [
    SystemMessage(content="You are a 7 years of java developer"),
    HumanMessage(content="How to find second min number in array")
]

cppPromt = [
    SystemMessage(content="You are a 7 years of c++ developer"),
    HumanMessage(content="How to map an array")
]

In [ ]:
# we can use ChatPromptTemplate

from langchain_core.prompts import ChatPromptTemplate

promptTemplate = ChatPromptTemplate.from_messages([
    ("system", "You are a developer with 7 years of experience in {language}. Explain concepts clearly and provide examples in {language}."),
    ("human", "{question}")
])

# Python prompt
pythonPrompt = promptTemplate.invoke({
    "language": "Python",
    "question": "How to find the second largest number in an array?"
})

# Java prompt
javaPrompt = promptTemplate.invoke({
    "language": "Java",
    "question": "How to find the second minimum number in an array?"
})

# C++ prompt
cppPrompt = promptTemplate.invoke({
    "language": "C++",
    "question": "How to map an array?"
})

# Invoke the LLM
pythonResponse = llm.invoke(pythonPrompt)
javaResponse = llm.invoke(javaPrompt)
cppResponse = llm.invoke(cppPrompt)

# Print responses
print("=== Python Response ===")
print(pythonResponse.content)

print("\n=== Java Response ===")
print(javaResponse.content)

print("\n=== C++ Response ===")
print(cppResponse.content)


=== Python Response ===
## Finding the **second largest** number in an array (Python list)

Below you’ll find several ways to solve the problem, each with its own trade‑offs.  
I’ll walk through the logic, show the code, and discuss edge‑cases and performance.

---

### 1. The “two‑variable” single‑pass algorithm (O(n) time, O(1) space)

The most efficient way (in terms of both time and memory) is to keep track of the largest and the second largest values while iterating once through the list.

```python
def second_largest(nums):
    if len(nums) < 2:
        raise ValueError("Array must contain at least two elements")

    # Initialise with the first two elements in the correct order
    first = second = float('-inf')
    for n in nums:
        if n > first:
            second = first   # old first becomes second
            first = n
        elif n > second and n != first:   # skip duplicates of the max
            second = n

    if second == float('-inf'):
        raise ValueError(

### LCEL(LangChain Expression Language) - `|` operator

In [ ]:
chain = promptTemplate | llm #We can make multiple chains

response = chain.invoke({
    "language": "Python",
    "question": "Explain decorators in 5 sentennce."
})

print(response.content)

Decorators are higher‑order functions that take another function (or class) as an argument and return a new function, allowing you to modify or extend behavior without changing the original code. They are applied with the `@decorator_name` syntax just above the function definition, which is syntactic sugar for `function = decorator_name(function)`. A common use case is logging, timing, or access control, where the decorator wraps the original function, executes pre‑ or post‑logic, and then calls the wrapped function. Because decorators preserve the wrapped function’s signature and metadata (via `functools.wraps`), they can be composed and reused across modules. Example:  

```python
import time, functools

def timer(func):
    @functools.wraps(func)
    def wrapper(*args, **kwargs):
        start = time.perf_counter()
        result = func(*args, **kwargs)
        print(f"{func.__name__} took {time.perf_counter() - start:.4f}s")
        return result
    return wrapper

@timer
def comp

#### `Memory` - coversational history

In [29]:
from langchain_core.messages import HumanMessage, AIMessage

chat_history = []

print("Chatbot started! Enter 0 to exit.\n")

while True:
    user_input = input("You: ")

    if user_input.strip() == "0":
        print("Chatbot: Goodbye, Akash!")
        break

    chat_history.append(HumanMessage(content=user_input))

    response = llm.invoke(chat_history)

    print("AI:", response.content, "\n")

    chat_history.append(AIMessage(content=response.content))

Chatbot started! Enter 0 to exit.

AI: Hello! How can I help you today? 

AI: Nice to meet you, Akash! How can I assist you today? 

AI: Your name is Akash. 

Chatbot: Goodbye, Akash!


In [30]:
chat_history

[HumanMessage(content='hi', additional_kwargs={}, response_metadata={}),
 AIMessage(content='Hello! How can I help you today?', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]),
 HumanMessage(content='My name is Akash.', additional_kwargs={}, response_metadata={}),
 AIMessage(content='Nice to meet you, Akash! How can I assist you today?', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]),
 HumanMessage(content='what is my name?', additional_kwargs={}, response_metadata={}),
 AIMessage(content='Your name is Akash.', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[])]

### Structured output

In [31]:
from pydantic import BaseModel, Field

class Developer(BaseModel):
    name: str = Field(description="Developer's name")
    primary_language: str = Field(
        description="Developer's main programming language"
    )
    experience_level: str = Field(
        description="Beginner, intermediate, or advanced"
    )
    interests: list[str] = Field(
        description="Developer's technical interests"
    )

# Create an LLM that returns structured output
structured_llm = llm.with_structured_output(Developer)

# Get structured information from the LLM
response = structured_llm.invoke(
    """
    My name is Akash. I mainly use Python.
    I am still learning AI engineering.
    My interests are backend development, FastAPI,
    large language models, and AI agents.
    """
)

# Print the complete result
print("Complete structured response:")
print(response)

# Access individual fields
print("\nDeveloper Details:")
print("Name:", response.name)
print("Primary Language:", response.primary_language)
print("Experience Level:", response.experience_level)
print("Interests:", response.interests)

# Convert the Pydantic object to a dictionary
print("\nDictionary:")
print(response.model_dump())

# Convert the result to JSON
print("\nJSON:")
print(response.model_dump_json(indent=2))


Complete structured response:
name='Akash' primary_language='Python' experience_level='intermediate' interests=['backend development', 'FastAPI', 'large language models', 'AI agents']

Developer Details:
Name: Akash
Primary Language: Python
Experience Level: intermediate
Interests: ['backend development', 'FastAPI', 'large language models', 'AI agents']

Dictionary:
{'name': 'Akash', 'primary_language': 'Python', 'experience_level': 'intermediate', 'interests': ['backend development', 'FastAPI', 'large language models', 'AI agents']}

JSON:
{
  "name": "Akash",
  "primary_language": "Python",
  "experience_level": "intermediate",
  "interests": [
    "backend development",
    "FastAPI",
    "large language models",
    "AI agents"
  ]
}


### AI Agents